# 🏁 Projeto Final — o ciclo completo de MLOps num só fluxo

Acompanha o **Módulo 7** do guia. Aqui você encadeia, de ponta a ponta, o ciclo
de vida de um modelo — que é exatamente o que o MLOps organiza:

```
1. Dados  →  2. Baseline  →  3. Fine-tuning (LoRA)  →  4. Tracking (MLflow)
          →  5. Avaliação  →  6. Empacotamento (fuse)  →  7. Relatório
```

> Abra com `make lab` a partir da raiz, com o `.venv` ativo.
> Ajuste `ITERS` conforme o tempo que quiser investir.

## 1. Configuração e dados

Garantimos que os dados existem (senão geramos com o script 01).

In [ ]:
import os
import re
import sys
import subprocess
import shutil

import matplotlib.pyplot as plt
import mlflow

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)

MODEL = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
ADAPTER = "models/finetuned/projeto-final-adapters"
MERGED = "models/finetuned/projeto-final-merged"
ITERS = 100
PERGUNTAS = ["O que é MLOps?", "O que é LoRA?", "O que é quantização?"]

if not os.path.exists("data/processed/train.jsonl"):
    subprocess.run([sys.executable, "scripts/01_prepare_data.py"], check=True)

n_train = sum(1 for _ in open("data/processed/train.jsonl"))
n_valid = sum(1 for _ in open("data/processed/valid.jsonl"))
print(f"Dados prontos: {n_train} treino / {n_valid} validação")
print(f"Modelo base: {MODEL}")

## 2. Baseline — respostas do modelo ANTES do treino

In [ ]:
def responder(prompt, adapter=None, max_tokens=70):
    cmd = [sys.executable, "-m", "mlx_lm", "generate", "--model", MODEL,
           "--prompt", prompt, "--max-tokens", str(max_tokens), "--temp", "0.3"]
    if adapter:
        cmd += ["--adapter-path", adapter]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    parts = out.split("==========")
    return parts[1].strip() if len(parts) >= 2 else out.strip()

baseline = {q: responder(q) for q in PERGUNTAS}
for q, a in baseline.items():
    print(f"P: {q}\nR (base): {a}\n")

## 3 + 4. Fine-tuning LoRA COM tracking no MLflow

Rodamos o treino, capturamos as métricas do stdout e as registramos no MLflow —
amarrando o **treino real** ao **experiment tracking**. Este é o coração do MLOps.

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(ROOT, 'mlflow.db')}")
mlflow.set_experiment("projeto-final")

shutil.rmtree(ADAPTER, ignore_errors=True)
cmd = [sys.executable, "-m", "mlx_lm", "lora", "--model", MODEL, "--train",
       "--data", "data/processed", "--adapter-path", ADAPTER,
       "--iters", str(ITERS), "--batch-size", "4", "--num-layers", "8",
       "--learning-rate", "1e-4", "--steps-per-report", "10",
       "--steps-per-eval", "25"]

re_train = re.compile(r"Iter (\d+): Train loss ([\d.]+)")
re_val = re.compile(r"Iter (\d+): Val loss ([\d.]+)")
tx, ty, vx, vy = [], [], [], []

with mlflow.start_run(run_name=f"lora-{ITERS}iters"):
    mlflow.log_params({"model": MODEL, "iters": ITERS, "lr": 1e-4,
                       "batch_size": 4, "num_layers": 8, "method": "lora"})
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        mt, mv = re_train.search(line), re_val.search(line)
        if mt:
            it, v = int(mt.group(1)), float(mt.group(2))
            tx.append(it); ty.append(v)
            mlflow.log_metric("train_loss", v, step=it)
            print(f"  treino iter {it:>4} | loss {v}")
        if mv:
            it, v = int(mv.group(1)), float(mv.group(2))
            vx.append(it); vy.append(v)
            mlflow.log_metric("val_loss", v, step=it)
    proc.wait()
    if vy:
        mlflow.log_metric("final_val_loss", vy[-1])
print("\nTreino + tracking concluídos.")

In [ ]:
plt.figure(figsize=(8, 4))
if tx: plt.plot(tx, ty, label="train", marker="o", ms=4)
if vx: plt.plot(vx, vy, label="val", marker="s", ms=4)
plt.title("Projeto final — curva de loss do LoRA")
plt.xlabel("iteração"); plt.ylabel("loss"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Avaliação — comparar respostas ANTES × DEPOIS

In [ ]:
for q in PERGUNTAS:
    depois = responder(q, adapter=ADAPTER)
    print("=" * 70)
    print("P:", q)
    print("\n  🔹 base :", baseline[q][:200])
    print("\n  🔸 LoRA :", depois[:200])
print("=" * 70)

## 6. Empacotamento — fundir o adaptador num modelo standalone

O `fuse` junta o LoRA ao modelo base, gerando um modelo independente (não
precisa mais carregar o adaptador à parte). É o artefato que você entregaria.

In [ ]:
shutil.rmtree(MERGED, ignore_errors=True)
subprocess.run([sys.executable, "-m", "mlx_lm", "fuse",
                "--model", MODEL, "--adapter-path", ADAPTER,
                "--save-path", MERGED], check=True)

files = os.listdir(MERGED)
print("Modelo standalone salvo em", MERGED)
print("Arquivos:", files)

## 7. Relatório final

Resumo do que foi produzido — e o próximo passo opcional (exportar para GGUF).

In [ ]:
print("RELATÓRIO DO PROJETO FINAL")
print("-" * 50)
print(f"Modelo base      : {MODEL}")
print(f"Exemplos treino  : {n_train}")
print(f"Iterações        : {ITERS}")
if vy:
    print(f"Val loss inicial : {vy[0]:.3f}")
    print(f"Val loss final   : {vy[-1]:.3f}")
print(f"Adaptador LoRA   : {ADAPTER}")
print(f"Modelo fundido   : {MERGED}")
print("Tracking         : MLflow, experimento 'projeto-final' (make mlflow)")
print("-" * 50)
print("\nPróximo passo opcional — exportar para GGUF (rodar no Ollama/llama.cpp):")
print(f"  brew install llama.cpp")
print(f"  python convert_hf_to_gguf.py {MERGED} \\")
print(f"      --outfile models/gguf/projeto-final.gguf --outtype q8_0")

## 🎓 Parabéns!

Você percorreu o ciclo completo: **dados → baseline → treino → tracking →
avaliação → empacotamento**. Agora troque os dados por um domínio seu e repita.

**Ideias de projeto real:**
- Um assistente de FAQ sobre um produto/tema que você domina
- Um reescritor de texto no seu estilo
- Um classificador de intenções

📝 Documente seu projeto no `docs/03_DIARIO.md`.